Import Necessary libraries

In [97]:
import pandas as pd
from pandas import DataFrame
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Read CSV and Explore important features.
- This datasets have only 3 types of data, first one is object/timeseries, second one is the float dtype which is the features value, and the third one is interger dtype which is the class to predict.
- The class only have 2 unique value which is -1 (pass) and 1 (fail). we can also notice that the class is extremely imbalance.

In [98]:
df = pd.read_csv('uci-secom.csv')
print(f"Datasets Shape: {df.shape}")
print(f'Unique Dtypes: {df.dtypes.unique()}')
print(f'Column to predict: {df.columns[-1]}')
print(f'Class: {df['Pass/Fail'].unique()}')
print(f'Class count: {df['Pass/Fail'].value_counts()}')
print(f'Missing values: {df.isna().sum().sum()}')

Datasets Shape: (1567, 592)
Unique Dtypes: [dtype('O') dtype('float64') dtype('int64')]
Column to predict: Pass/Fail
Class: [-1  1]
Class count: Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64
Missing values: 41951


We are going to start dividing the datasets into features (X) and classes (y).
Next we are going to replace the classes value with a more common value (0 and 1).

In [99]:
X = df.iloc[:, 0:-1]
y = df.iloc[:, -1]
y = y.replace([-1, 1], [0, 1])

Replace missing value with mean, and dropping rows with too many missing features.

In [100]:
def handle_missing(data: DataFrame) -> tuple[DataFrame, list[int]]:
    data = data.copy()
    data = data.drop(columns = 'Time', errors='ignore')
    # if in a row, more than 100 of its features have missing values, we are going to drop the row.
    # else we fill it with the mean.
    na_value = data.isna().sum(axis=1)
    to_drop = []
    for index, value in enumerate(na_value):
        if value > 100:
            to_drop.append(index)
    data = data.drop(to_drop, axis=0)
    for column in data.columns:
        data[column] = data[column].fillna(data[column].median())
    return data, to_drop
X, to_drop = handle_missing(X)
y = y.drop(to_drop, axis=0)

- Import all necessary libraries for machine learning
- Splitting data into training and testing sets.

In [106]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

- Build a function to evaluate the best K for SelectKBest to filter out how many best classes would result in a better f1 score.
- Applying Variance Threshold and Standard Scaler to result in better model perfomance.
- Implementing mlflow for easier logging and visualization.

In [107]:
import mlflow
from sklearn.model_selection import cross_val_score
def evaluate_k(k: list[int] | None = None):
    if k is None:
        k = [10, 25, 50, 100, 200]
    for i in k:
        my_pipe = Pipeline([
            ('variance', VarianceThreshold()),
            ('selector', SelectKBest(score_func=mutual_info_classif, k=i)),
            ('scaler', StandardScaler()),
            ('model', RandomForestClassifier(class_weight='balanced'))
        ])
        scores = cross_val_score(my_pipe, X, y, scoring='f1', cv=5)
        with mlflow.start_run():
            mlflow.log_param('k_value', i)
            mlflow.log_metric('mean_f1', scores.mean())
            mlflow.log_metric('std_f1', scores.std())

evaluate_k([3, 5, 7, 10, 15])

- Build a function to  evaluate best parameter to use on the model (n_estimator and max_depth)
- Implementing mlflow for easier logging and visualization

In [ ]:
def evaluate_param(params: list[dict]):
    for param in params: 
        my_pipe = Pipeline([
            ('variance', VarianceThreshold()),
            ('selector', SelectKBest(score_func=f_classif, k=10)),
            ('scaler', StandardScaler()),
            ('model', RandomForestClassifier(class_weight='balanced', **param))
        ])
        scores = cross_val_score(my_pipe, X, y, scoring='f1', cv=5)

        with mlflow.start_run():
            for key, val in param.items():
                mlflow.log_param(key, val)
            mlflow.log_metric('mean_f1', scores.mean())
            mlflow.log_metric('std_f1', scores.std())

params = [
    {'n_estimators': 100, 'max_depth': 5},
    {'n_estimators': 200, 'max_depth': 5},
    {'n_estimators': 300, 'max_depth': 5},
    {'n_estimators': 200, 'max_depth': 7},
    {'n_estimators': 200, 'max_depth': 10},
]

evaluate_param(params)